# Lab Assignment 04: Implementing U-Net for Image Segmentation

**Course:** CSET-225 - IMDAI  
**Program/Semester:** B.Tech., Semester 5  
**Student:** Rupesh Sharma  
**Dataset:** Oxford-IIIT Pet Dataset

This notebook covers the single-image forward-pass demonstration, complete U-Net implementation, training, evaluation using pixel accuracy and IoU, learning curves, and qualitative prediction results.

## How to run

Use **Runtime > Run all** in Google Colab. GPU is recommended. The default settings use a manageable subset and 8 epochs; increase `TRAIN_LIMIT`, `VAL_LIMIT`, and `EPOCHS` for a stronger final model.

In [ ]:
# Install only if the runtime does not already provide the packages.
!pip -q install tensorflow-datasets

In [ ]:
import os, random, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

SEED = 42
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 8
TRAIN_LIMIT = 2000   # Set to None to use the complete training split
VAL_LIMIT = 500      # Set to None to use the complete test split
NUM_CLASSES = 3      # pet, background, border

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

## 1. Load and inspect the Oxford-IIIT Pet dataset

The trimap mask contains values 1, 2, and 3. During preprocessing, they are converted to zero-based class IDs 0, 1, and 2, as required by sparse categorical cross-entropy.

In [ ]:
(raw_train, raw_test), info = tfds.load(
    'oxford_iiit_pet:3.*.*',
    split=['train', 'test'],
    with_info=True,
    shuffle_files=True
)
print(info)
print("Training examples:", info.splits['train'].num_examples)
print("Test examples:", info.splits['test'].num_examples)

In [ ]:
sample = next(iter(raw_train.take(1)))
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample['image'])
plt.title('Original image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(sample['segmentation_mask'][:, :, 0], cmap='viridis')
plt.title('Original trimap mask')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Preprocessing and augmentation

- Images are resized to 128 x 128 and normalized to [0, 1].
- Masks use nearest-neighbor resizing so class IDs are not interpolated.
- The image and mask are flipped together during training.

In [ ]:
def preprocess(example):
    image = tf.image.resize(example['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0

    mask = tf.image.resize(
        example['segmentation_mask'],
        (IMG_SIZE, IMG_SIZE),
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )
    mask = tf.cast(mask, tf.int32) - 1
    return image, mask

def augment(image, mask):
    do_flip = tf.random.uniform(()) > 0.5
    image = tf.cond(do_flip, lambda: tf.image.flip_left_right(image), lambda: image)
    mask = tf.cond(do_flip, lambda: tf.image.flip_left_right(mask), lambda: mask)
    return image, mask

train_source = raw_train.take(TRAIN_LIMIT) if TRAIN_LIMIT else raw_train
val_source = raw_test.take(VAL_LIMIT) if VAL_LIMIT else raw_test

train_ds = (train_source
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .map(augment, num_parallel_calls=AUTOTUNE)
            .cache()
            .shuffle(1000, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (val_source
          .map(preprocess, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .cache()
          .prefetch(AUTOTUNE))

images, masks = next(iter(train_ds))
print("Image batch:", images.shape, images.dtype)
print("Mask batch:", masks.shape, masks.dtype)
print("Mask classes:", np.unique(masks.numpy()))

## 3. U-Net architecture

The encoder extracts increasingly abstract features while reducing spatial size. The bottleneck learns the deepest representation. The decoder restores resolution. Skip connections concatenate encoder features with matching decoder stages, preserving boundary and location information.

In [ ]:
def conv_block(x, filters, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

def encoder_block(x, filters, dropout=0.0):
    skip = conv_block(x, filters, dropout)
    pooled = layers.MaxPooling2D(2)(skip)
    return skip, pooled

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    return conv_block(x, filters)

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(input_shape)
    s1, p1 = encoder_block(inputs, 32)
    s2, p2 = encoder_block(p1, 64)
    s3, p3 = encoder_block(p2, 128)
    s4, p4 = encoder_block(p3, 256, dropout=0.1)

    bridge = conv_block(p4, 512, dropout=0.3)

    d1 = decoder_block(bridge, s4, 256)
    d2 = decoder_block(d1, s3, 128)
    d3 = decoder_block(d2, s2, 64)
    d4 = decoder_block(d3, s1, 32)
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(d4)
    return models.Model(inputs, outputs, name='U_Net')

model = build_unet()
model.summary()

## 4. Single-image U-Net forward-pass demonstration

This satisfies the first task before training. The predicted mask is initially random because the network weights have not yet learned from the dataset.

In [ ]:
one_image, one_mask = next(iter(val_ds.unbatch().take(1)))
untrained_prob = model.predict(one_image[None, ...], verbose=0)[0]
untrained_mask = tf.argmax(untrained_prob, axis=-1)

plt.figure(figsize=(12, 4))
for i, (item, title) in enumerate([
    (one_image, 'Input image'),
    (one_mask[:, :, 0], 'True mask'),
    (untrained_mask, 'Untrained U-Net output')
]):
    plt.subplot(1, 3, i + 1)
    plt.imshow(item, cmap=None if i == 0 else 'viridis', vmin=0 if i else None, vmax=2 if i else None)
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 5. IoU metric and model training

Pixel accuracy measures the proportion of correctly classified pixels. Mean IoU is stricter: for each class it divides intersection by union, then averages across classes.

In [ ]:
class SparseMeanIoU(tf.keras.metrics.MeanIoU):
    def __init__(self, num_classes, name='mean_iou', **kwargs):
        super().__init__(num_classes=num_classes, name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=-1)
        y_true = tf.squeeze(tf.cast(y_true, tf.int32), axis=-1)
        return super().update_state(y_true, y_pred, sample_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
             SparseMeanIoU(NUM_CLASSES)]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_unet_pet.keras', monitor='val_mean_iou', mode='max', save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True
    )
]

start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)
print(f"Training time: {(time.time() - start) / 60:.2f} minutes")

## 6. Training and validation curves

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(1, len(h['loss']) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    plots = [('loss', 'Loss'), ('accuracy', 'Pixel accuracy'), ('mean_iou', 'Mean IoU')]
    for axis, (key, label) in zip(ax, plots):
        axis.plot(epochs, h[key], marker='o', label='Training')
        axis.plot(epochs, h['val_' + key], marker='o', label='Validation')
        axis.set_title(label)
        axis.set_xlabel('Epoch')
        axis.grid(alpha=0.3)
        axis.legend()
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=180, bbox_inches='tight')
    plt.show()

plot_history(history)

## 7. Quantitative evaluation and confusion matrix

In [ ]:
results = model.evaluate(val_ds, return_dict=True, verbose=1)
print({k: round(float(v), 4) for k, v in results.items()})

confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
for batch_images, batch_masks in val_ds:
    predictions = tf.argmax(model(batch_images, training=False), axis=-1)
    truth = tf.squeeze(batch_masks, axis=-1)
    confusion += tf.math.confusion_matrix(
        tf.reshape(truth, [-1]),
        tf.reshape(predictions, [-1]),
        num_classes=NUM_CLASSES
    ).numpy()

print("Confusion matrix (rows=true, columns=predicted):")
print(confusion)
class_iou = np.diag(confusion) / (
    confusion.sum(axis=1) + confusion.sum(axis=0) - np.diag(confusion) + 1e-7
)
for label, value in zip(['pet', 'background', 'border'], class_iou):
    print(f"IoU - {label}: {value:.4f}")
print(f"Macro mean IoU: {class_iou.mean():.4f}")

## 8. Visualize predictions

In [ ]:
def show_predictions(dataset, count=4):
    plt.figure(figsize=(12, 3 * count))
    for row, (image, true_mask) in enumerate(dataset.unbatch().take(count)):
        pred = model.predict(image[None, ...], verbose=0)[0]
        pred_mask = tf.argmax(pred, axis=-1)
        panels = [image, true_mask[:, :, 0], pred_mask]
        titles = ['Original image', 'Ground-truth mask', 'Predicted mask']
        for col in range(3):
            plt.subplot(count, 3, row * 3 + col + 1)
            plt.imshow(panels[col], cmap=None if col == 0 else 'viridis',
                       vmin=0 if col else None, vmax=2 if col else None)
            plt.title(titles[col])
            plt.axis('off')
    plt.tight_layout()
    plt.savefig('sample_predictions.png', dpi=180, bbox_inches='tight')
    plt.show()

show_predictions(val_ds, count=4)

## 9. Save the model and run summary

In [ ]:
model.save('unet_oxford_pet_final.keras')
best_epoch = int(np.argmax(history.history['val_mean_iou'])) + 1
print('Best validation-IoU epoch:', best_epoch)
print('Best validation IoU:', max(history.history['val_mean_iou']))
print('Final validation loss:', history.history['val_loss'][-1])

## Observations and conclusion

1. Before training, the predicted mask has no semantic meaning because weights are randomly initialized.
2. Skip connections improve localization by restoring high-resolution encoder features during decoding.
3. Accuracy may appear high when background pixels dominate, so mean IoU is a more informative segmentation metric.
4. The border class is usually harder than pet/background because it is thin and occupies fewer pixels.
5. Data augmentation, more epochs, and the full dataset generally improve robustness, while dropout and batch normalization reduce overfitting.

**Conclusion:** The implemented U-Net performs multiclass semantic segmentation of pet images into pet, background, and border regions. Quantitative metrics and qualitative masks together provide a reliable evaluation.